# To Bugfix

prepare.py        | steps.py                  | pipeline/                | status
--------------------------------------------------------------------------
prepare_dataset() | download()                | build_manifest()         | X
                  |                           | download_methylation()   | X
                  |                           | initialize_audit_table() | X
                  |                           | build_metadata()         | X
                  |                           | build_biospecimen()      | X
                  |                           | update_metadata()        | X
                  |                           | update_biospecimen()     | X
                  | clean_data()              |                          | X
                  | load_raw_project()        |                          | X
                  | quality_control()         | load_annotation()        | X
                  |                           | sample_qc()              | X
                  |                           | probe_qc()               | X
                  |                           | clean_metadata()         | X
                  | preprocess()              | normalize()              | X
                  |                           | filter_variance()        | X
                  |                           | convert_to_mval()        | X
                  |                           | impute()                 | X
                  | save_project()            |                          | X
prepare_cohort()  | load_processed_project()  |                          |
                  | aggregate_cohort()        | cohort_aggregation()     |
                  | cohort_batch_correction() | batch_correct()          |
                  | aggregate_genes()         | gene_aggregation()       |
                  | winsorize()               |                          |
                  | split()                   |                          |

%load_ext autoreload
%autoreload 2


## Initial Set-up

In [27]:
from methyltrain.config.loader import load_config
from methyltrain.fs.layout import ProjectLayout, CohortLayout

layout = ProjectLayout(
    project_name = "TCGA-ESCA",
    root_dir = "../data",
    raw_dir = "../data/raw/TCGA-ESCA",
    audit_table = "../data/metadata/TCGA-ESCA/TCGA-ESCA_audit_table.csv",
    metadata = "../data/metadata/TCGA-ESCA/TCGA-ESCA_metadata.csv",
    manifest = "../data/metadata/TCGA-ESCA/TCGA-ESCA_manifest.csv",
    status_log = "../data/metadata/TCGA-ESCA/TCGA-ESCA_status_log.csv",
    project_adata = "../data/processed/TCGA-ESCA_adata.h5ad"
)
layout.initialize()
layout.validate()

config = load_config(
    "/Volumes/FBI_Drive/MethylTrain/config/local_config.yaml"
)

## Current Debug

In [28]:
from typing import Dict, Tuple

import anndata as ad
import pandas as pd

from methyltrain.fs.layout import ProjectLayout, CohortLayout
from methyltrain.api.steps import (
    download, 
    clean_data, 
    quality_control, 
    preprocess, 
    aggregate_cohort, 
    cohort_batch_correction,
    aggregate_genes,
    winsorize,
    split,
    load_raw_project,
    load_processed_project
)

from methyltrain.pipeline.preprocess import (
    normalize, filter_variance, impute, convert_to_mval
)
from methyltrain.utils.load_utils import load_audit_table

In [26]:
import anndata as ad

adata = ad.read_h5ad("/Volumes/FBI_Drive/MethylTrain/data/processed/TCGA-OV_adata.h5ad")
adata

AnnData object with n_obs × n_vars = 8 × 301409
    obs: 'file_name', 'data_type', 'data_category', 'experimental_strategy', 'platform', 'project_id', 'submitter_id', 'sample_type', 'aliquot_id', 'status', 'missing_rate', 'mean'
    var: 'missing_rate', 'variance', 'frac_imputed', 'impute_value'
    uns: 'conversion', 'data_type', 'level', 'platform', 'preprocessing_steps', 'project_id', 'reference_genome', 'split', 'split_percentage', 'state'

In [37]:
from methyltrain.pipeline.download import (
    build_manifest, 
    download_methylation,
    build_metadata
)
from methyltrain.pipeline.audit import (
    initialize_audit_table, update_metadata
)

In [30]:
manifest = build_manifest(config)

In [33]:
manifest = manifest[1:5]

In [36]:
status_log = download_methylation(manifest, config, layout, verbose = True)

=====| Finished downloading TCGA-ESCA DNA Methylation Data |=====


In [38]:
audit_table = initialize_audit_table(manifest, status_log)

In [40]:
metadata = build_metadata(audit_table, config)

In [42]:
audit_table = update_metadata(audit_table, metadata)

In [43]:
metadata = metadata.loc[metadata['status'] == 'success']

In [44]:
from methyltrain.utils.load_utils import save_audit_table, save_manifest, save_metadata, save_status_log

save_manifest(manifest, layout)
save_status_log(status_log, layout)
save_metadata(metadata, layout)
save_audit_table(audit_table, layout)

In [45]:
audit_table = clean_data(audit_table, layout)

In [47]:
adata = load_raw_project(config, layout)

In [48]:
adata

AnnData object with n_obs × n_vars = 4 × 486426
    obs: 'file_name', 'data_type', 'data_category', 'experimental_strategy', 'platform', 'project_id', 'submitter_id', 'sample_type', 'aliquot_id', 'status'
    uns: 'project_id', 'platform', 'reference_genome', 'level', 'data_type', 'conversion', 'state', 'preprocessing_steps', 'split', 'split_percentage'

In [49]:
metadata

,file_name,data_type,data_category,experimental_strategy,platform,project_id,submitter_id,sample_type,aliquot_id,status
file_id,,,,,,,,,,
350a363d-ba9a-4068-9e67-cd35ab3256f2,15649376-2f18-49b1-a9c0-67ce1bb8ee01.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-ESCA,TCGA-LN-A5U5,Primary Tumor,ec36ec4a-1956-4196-a453-29c89f93eaf1,success
2911f973-3b2c-4061-a909-15f5f68284c6,7ff5051c-8fa9-4f3f-b4b2-aef45c143a2d.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-ESCA,TCGA-LN-A49Y,Primary Tumor,d3d01346-760d-4c73-8e50-9349d146df7f,success
a8b9be4b-27cd-46a2-b0e6-1e9158ab97a6,25f6bba7-c299-46de-ac53-c3cb30e8026d.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-ESCA,TCGA-LN-A5U6,Primary Tumor,bb634edb-2bd0-4a6f-831e-84211309fe40,success
dfd270af-30b3-45b6-8835-452c17cf4d77,5d819338-9703-4d60-a37a-d197fbfa0594.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-ESCA,TCGA-R6-A6DN,Primary Tumor,e49cb14f-8370-49d8-b6de-f8ac1f676c79,success
